In [1]:
# Cell 1: Setup and Transfer Model Compiling
import sys
from pathlib import Path

# Add project root to python path
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.dataset import get_data_loaders
from src.models import build_transfer_learning_model
from src.trainer import compile_and_get_callbacks

# 1. Load Data
train_ds, val_ds, test_ds = get_data_loaders()

# 2. Build Transfer Model
transfer_model = build_transfer_learning_model()

# 3. Look at how many parameters are actually trainable vs frozen!
transfer_model.summary()

Initializing Data Pipelines...
Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Found 3000 files belonging to 6 classes.
Loading pre-trained MobileNetV2 Base...


c:\Users\indra\OneDrive\Pictures\DeepTrain\DeepTrain\src\models.py:45: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Model: "Transfer_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [2]:
import sys
from pathlib import Path

# Add project root to python path first
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

# Now we can import from src!
from src.trainer import compile_and_get_callbacks

# 1. Reconstruct the baseline model's history from the 04_Training.ipynb outputs
class MockHistory:
    def __init__(self, history_dict):
        self.history = history_dict

# These are the exact metrics logged in your 04_Training.ipynb run
history = MockHistory({
    'accuracy': [0.5888, 0.6978, 0.7615, 0.8139, 0.8425],
    'loss': [1.0655, 0.8156, 0.6648, 0.5289, 0.4506],
    'val_accuracy': [0.6914, 0.7441, 0.7798, 0.8033, 0.8115],
    'val_loss': [0.7998, 0.7023, 0.5906, 0.5433, 0.5500]
})

# 2. Train the Transfer Learning Model
EPOCHS = 5
print("Starting transfer model training loop...")

# You can reuse compile_and_get_callbacks here if you want it to save checkpoints
# Note: we use a lower learning rate (e.g., 1e-4) for transfer learning usually
callbacks = compile_and_get_callbacks(transfer_model, learning_rate=0.0001)

history_transfer = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

Starting transfer model training loop...
Model compiled. Checkpoints will be saved to: c:\Users\indra\OneDrive\Pictures\DeepTrain\DeepTrain\outputs\checkpoints\best_baseline_model.keras
Epoch 1/5


351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.4464 - loss: 1.5029
Epoch 1: val_loss improved from None to 0.51064, saving model to c:\Users\indra\OneDrive\Pictures\DeepTrain\DeepTrain\outputs\checkpoints\best_baseline_model.keras
351/351 ━━━━━━━━━━━━━━━━━━━━ 134s 332ms/step - accuracy: 0.6096 - loss: 1.0605 - val_accuracy: 0.8343 - val_loss: 0.5106
Epoch 2/5
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step - accuracy: 0.8105 - loss: 0.5232
Epoch 2: val_loss improved from 0.51064 to 0.37231, saving model to c:\Users\indra\OneDrive\Pictures\DeepTrain\DeepTrain\outputs\checkpoints\best_baseline_model.keras
351/351 ━━━━━━━━━━━━━━━━━━━━ 113s 322ms/step - accuracy: 0.8234 - loss: 0.4891 - val_accuracy: 0.8753 - val_loss: 0.3723
Epoch 3/5
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step - accuracy: 0.8492 - loss: 0.4128
Epoch 3: val_loss improved from 0.37231 to 0.33154, saving model to c:\Users\indra\OneDrive\Pictures\DeepTrain\DeepTrain\outputs\checkpoints\best_baseline_model.keras
351/351 ━

In [3]:
# Final Reporting Cell
import sys
from pathlib import Path

# Add project root to python path
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.utils import generate_framework_report

# Generate and display the final performance matrix
report_df, styled_table = generate_framework_report(history, history_transfer)

# Show the table beautifully inside VS Code
styled_table


Compiling DeepTrain Framework Performance Report...


Experiment Model,Peak Train Acc,Peak Val Acc,Final Val Loss,Architecture Style
Baseline CNN (From Scratch),84.25%,81.15%,0.5500,Custom Conv2D + MaxPooling
MobileNetV2 (Transfer Learning),87.92%,89.88%,0.2964,Frozen Base + Custom Dense Head
